# Machine Learning
In this notebook, we will be training Machine Learning models to predict the default flag of a given loan. We will be creating a pseudo-label using the loan_status column by classifying extremely late, charged off, and default loans as having a True default flag. We will be using Spark ML to train our models to take advantage of distributed computation and make our approach scaleable for big data.

We will be training two separate models in this phase of our project, known as the dev phase. Based on their performance on an unseen dataset in the next phase, one of these two models will be selected for inference on entirely new data.

#### Import and Read the data

In [0]:
spark.conf.set(
    "fs.azure.account.key.lcdevadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcstgadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcprodadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lendingclubstorage.blob.core.windows.net",
    "REDACTED"
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, lit, sum
import pyspark.sql.functions as F

# INPUT_PATH = "../dataset/processed"
INPUT_PATH = "wasbs://lcloansdev@lcdevadls.blob.core.windows.net/"

df = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

Total rows: 1,321,846
+----+---------+---------+-----------+---------------+----+--------+-----------+-----+---------+--------------------+----------+--------------+----------+-------------------+----------+-----------+----------+----+----+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+----------------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+---

#### Create the default_flag Column

In [0]:
# For default flag
df = df.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

#### Observe any Class Imabalances

In [0]:
# Let's also check the overall default rate
df.agg(
    count("*").alias("total_loans"),
    sum("default_flag").alias("total_defaults"),
).withColumn(
    "default_rate",
    when(col("total_loans") != 0, col("total_defaults") / col("total_loans")).otherwise(None)
).show()

+-----------+--------------+------------------+
|total_loans|total_defaults|      default_rate|
+-----------+--------------+------------------+
|    1321846|        225268|0.1704192470227243|
+-----------+--------------+------------------+



We have a significant class imbalance in our dataset as we have many more non-default loans as we do defaults. To mitigate this, we will be taking two steps:
- We will be using the professional technique of using Class Weights. The minority class will be assigned a higher weight during training, allowing us to train the model properly. This technique is superior to more naive techniques like oversampling the minority task as it retains the original distribution.
- We won't rely on accuracy as a tool to evaluate our models. Due to this class imabalance (which plausibly also exists in the testing dataset), accuracy will be misleading as the model could always predict the majority class and still score a >95% accuracy. 
- Instead of accuracy, we will be looking at the AUC-ROC value which measures how well the model ranks positives vs negatives, and is known to be great for binary classification. 
- While AUC-ROC will be our main focus, we will also report F1 score (another good metric in this case) and accuracy.

#### Compute Class Weights

In [0]:
counts = df.groupBy("default_flag").count().collect()
count_dict = {row["default_flag"]: row["count"] for row in counts}
total = __builtins__.sum(count_dict.values())
weight_default = total / (2 * count_dict.get(1, 1))
weight_non_default = total / (2 * count_dict.get(0, 1))

df = df.withColumn(
    "class_weight",
    when(col("default_flag") == 1, weight_default).otherwise(weight_non_default)
)

#### Prepare Features
- Select the Numeric and Categorical features we want to use
- Clean and caste them
- Impute missing values
- Encode the categorical features for training
- Combine the features using VectorAssembler

In [0]:
from pyspark.sql.functions import col, when, trim
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, Imputer, StandardScaler
)
from pyspark.ml import Pipeline

# Define feature columns
numeric_cols = ["loan_amnt", "int_rate", "installment",
                "dti", "inq_last_6mths", "open_acc", "revol_bal",
                "revol_util", "bc_util"]

categorical_cols = ["term", "grade", "sub_grade", "home_ownership",
                    "verification_status", "purpose", "addr_state"]

# Select only needed columns for modeling
df_model = df.select(
    "default_flag", "class_weight",
    *numeric_cols, 
    *categorical_cols
)

# Clean categorical values - replace empty strings and nulls with "Unknown"
for c in categorical_cols:
    df_model = df_model.withColumn(
        c,
        when((col(c).isNull()) | (trim(col(c)) == ""), "Unknown").otherwise(trim(col(c)))
    )

# Clean numeric columns - replace invalid values with null before casting
for c in numeric_cols:
    df_model = df_model.withColumn(
        c,
        when(
            col(c).isNull() | (col(c).cast("string") == "") | (trim(col(c).cast("string")) == ""),
            None
        ).otherwise(col(c).cast("double"))
    )

# Impute missing numeric values with median
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[c + "_imp" for c in numeric_cols],
    strategy="median"
)

# Index categorical features (converts strings to numeric indices)
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep"  # Keeps unseen values in test set
    )
    for c in categorical_cols
]

# One-Hot Encode the indexed categorical features
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categorical_cols],
    outputCols=[c + "_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

# Assemble all features into a single vector
assembler = VectorAssembler(
    inputCols=[c + "_imp" for c in numeric_cols] +
              [c + "_ohe" for c in categorical_cols],
    outputCol="features",
    handleInvalid="keep"  # Keeps rows with NaN/null values
)

# Scale features for better model performance
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False  # Keep False for sparse vectors from OneHotEncoder
)

# Split data into training and test sets
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

print(f"Training set size: {train_df.count():,}")
print(f"Test set size: {test_df.count():,}")
print("\nFeature preparation complete!")
print(f"- {len(numeric_cols)} numeric features")
print(f"- {len(categorical_cols)} categorical features")

Training set size: 1,057,396
Test set size: 264,450

Feature preparation complete!
- 9 numeric features
- 7 categorical features


#### Model 1: Logistic Regression
##### Training

In [0]:
from pyspark.ml.classification import LogisticRegression

# Create Logistic Regression model
lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="default_flag",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01
)

# Create the complete pipeline with all stages
pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler, lr])

# Fit the model on training data
print("Training Logistic Regression model...")
model1 = pipeline.fit(train_df)
print("Model 1 (Logistic Regression) training complete!")

Training Logistic Regression model...
Model 1 (Logistic Regression) training complete!


##### Evaluation

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Make predictions
predictions = model1.transform(test_df)

# BinaryClassificationEvaluator for AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="default_flag", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc_roc_1 = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc_roc_1:.4f}")

# MulticlassClassificationEvaluator for F1
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="f1"
)
f1_score = evaluator_f1.evaluate(predictions)
print(f"F1-score: {f1_score:.4f}")

# Accuracy (not ideal for class imbalance, but can report)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")

AUC-ROC: 0.6926
F1-score: 0.6673
Accuracy: 0.6205


##### Save Model 1

#### Model 2: Random Forest Classifier
##### Training

In [0]:
from pyspark.ml.classification import RandomForestClassifier

# Create Random Forest Classifier
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="default_flag",
    weightCol="class_weight", 
    numTrees=50,
    maxDepth=5,
    seed=42
)

# Create the complete pipeline with all stages
pipeline = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler, rf])

# Fit the model on training data
print("Training Random Forest model...")
model2 = pipeline.fit(train_df)
print("Model 2 (Random Forest) training complete!")

Training Random Forest model...
Model 2 (Random Forest) training complete!


##### Evaluation

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Make predictions
predictions = model2.transform(test_df)

# BinaryClassificationEvaluator for AUC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="default_flag", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)
auc_roc = evaluator_auc.evaluate(predictions)
print(f"AUC-ROC: {auc_roc:.4f}")

# MulticlassClassificationEvaluator for F1
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="f1"
)
f1_score = evaluator_f1.evaluate(predictions)
print(f"F1-score: {f1_score:.4f}")

# Accuracy (not ideal for class imbalance, but can report)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="default_flag", 
    predictionCol="prediction", 
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)
print(f"Accuracy: {accuracy:.4f}")


AUC-ROC: 0.6811
F1-score: 0.6407
Accuracy: 0.5903


##### Read Data from staging to test both models

In [0]:
# Read data from staging to test both models

# INPUT_PATH = "../dataset/processed"
INPUT_PATH = "wasbs://lcloansstg@lcstgadls.blob.core.windows.net/"

df_test = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df_test.cache()

print(f"Total rows: {df_test.count():,}")

Total rows: 443,579


Create the default flag

In [0]:
# For default flag
df_test = df_test.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

In [0]:
# Select only needed columns for model testing
df_test = df_test.select(
    "default_flag",
    *numeric_cols, 
    *categorical_cols
)

# Clean categorical values - replace empty strings and nulls with "Unknown"
for c in categorical_cols:
    df_test = df_test.withColumn(
        c,
        when((col(c).isNull()) | (trim(col(c)) == ""), "Unknown").otherwise(trim(col(c)))
    )

# Clean numeric columns - replace invalid values with null before casting
for c in numeric_cols:
    df_test = df_test.withColumn(
        c,
        when(
            col(c).isNull() | (col(c).cast("string") == "") | (trim(col(c).cast("string")) == ""),
            None
        ).otherwise(col(c).cast("double"))
    )

In [0]:
# Run both model1 and model2 on this df
evaluator = BinaryClassificationEvaluator(
    labelCol="default_flag",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Predictions
preds1 = model1.transform(df_test)
preds2 = model2.transform(df_test)

# ROC-AUC
auc1 = evaluator.evaluate(preds1)
auc2 = evaluator.evaluate(preds2)

print(f"Model1 ROC-AUC: {auc1:.4f}")
print(f"Model2 ROC-AUC: {auc2:.4f}")

# Pick best model
best_model = model1 if auc1 >= auc2 else model2
best_name = "model1: Logistic Regression" if auc1 >= auc2 else "model2: Random Forest"
print(f"Best model: {best_name} with ROC-AUC={max(auc1, auc2):.4f}")


Model1 ROC-AUC: 0.6935
Model2 ROC-AUC: 0.6788
Best model: model1: Logistic Regression with ROC-AUC=0.6935


##### Save Best Model

In [0]:
import mlflow
import mlflow.spark
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

signature = ModelSignature(
    inputs=Schema([
        ColSpec("double", c) for c in numeric_cols
    ] + [
        ColSpec("string", c) for c in categorical_cols
    ]),
    outputs=Schema([
        ColSpec("double", "prediction"),
        ColSpec("string", "probability")
    ])
)

print("Signature:\n", signature)

Signature:
 inputs: 
  ['loan_amnt': double, 'int_rate': double, 'installment': double, 'dti': double, 'inq_last_6mths': double, 'open_acc': double, 'revol_bal': double, 'revol_util': double, 'bc_util': double, 'term': string, 'grade': string, 'sub_grade': string, 'home_ownership': string, 'verification_status': string, 'purpose': string, 'addr_state': string]
outputs: 
  ['prediction': double, 'probability': string]
params: 
  None



In [0]:

mlflow.set_experiment("/Shared/lc-loans-models")

with mlflow.start_run(run_name="loan_pipeline_model"):
    # log model
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="loan_pipeline_model",
        registered_model_name="LC_Loan_Pipeline",
        signature=signature
    )


2025/12/03 02:30:53 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when calling log_model().


2025/12/03 02:31:01 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false
2025/12/03 02:31:30 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/2789834236706630/bf63e906ad3c41c1bab4a72315d01f20/artifacts/loan_pipeline_model/sparkml, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.
/databricks/python/lib/python3.10/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

Registered model 'LC_Loan_Pipeline' already exists. Creating a new version of this model...


2025/12/03 02:31:39 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false


Uploading artifacts:   0%|          | 0/124 [00:00<?, ?it/s]

2025/12/03 02:31:40 INFO mlflow.store.artifact.cloud_artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false
Created version '5' of model 'lcdbfs_2231733496212994.default.lc_loan_pipeline'.
